In [ ]:
from sagemaker.image_uris import retrieve
import logging 
import sys

def setup_logger( logger_id ):
    logger = logging.getLogger( logger_id )
    logger.setLevel(logging.INFO)

    if not logger.handlers:
        handler = logging.StreamHandler(sys.stdout)
        formatter = logging.Formatter(
            f"%(asctime)s - {logger_id} - %(levelname)s - %(message)s",
            "%Y-%m-%d %H:%M:%S",
        )
        handler.setFormatter(formatter)
        logger.addHandler(handler)

    return logger

nb_logger = setup_logger("training_estimator")

image_uri = retrieve(
    framework="xgboost",
    region="ap-southeast-2",
    version="1.7-1"
)


In [ ]:
# Run for local docker environment

import os
from sagemaker.local import LocalSession

nb_logger.info(os.getcwd())
os.environ["AWS_PROFILE"] = "devopsiam"

session = LocalSession()
session.config = {
    "local": {
        "local_code": True        
        }    
    }
session._default_bucket = "local-bucket"

role = "arn:aws:iam::046576049723:role/service-role/AmazonSageMaker-ExecutionRole-20260703T120261"

In [ ]:
# aws
from sagemaker.estimator import Estimator
from sagemaker import get_execution_role

role = get_execution_role()

estimator = Estimator(
    image_uri=image_uri,
    entry_point="train.py",
    role=role,
    instance_type="ml.m5.large",
    instance_count=1,
    py_version="py3"
)

s3_path = "s3://amazon-sagemaker-046576049723-ap-southeast-2-6mbg5spiobazoi/shared/data_science/home_prices"


In [ ]:
# local

from sagemaker.estimator import Estimator

estimator = Estimator(
    image_uri=image_uri,
    role=role,                     
    instance_count=1,
    instance_type="local",
    entry_point="train.py",
    source_dir=".",
    sagemaker_session=session,
)

s3_path = "file://input/data/train"

In [ ]:
train_path = f"{s3_path}/house_prices_train.csv"
test_path = f"{s3_path}/house_prices_test.csv"

nb_logger.info("Verifying estimator variables...")
nb_logger.info(f"image_uri={estimator.image_uri}, role={estimator.role}")
nb_logger.info(f"train_path={train_path}, test_path={test_path}")

In [ ]:
nb_logger.info("Start training...")

estimator.fit({
    "train": train_path,
    "test": test_path
    })

nb_logger.info("Finished Training")